# 🚀 Career-Ops — Free LLM Server on Colab

Runs an open-source LLM on Colab's free T4 GPU and exposes an OpenAI-compatible API.

**Steps:** Verify GPU → Install → Configure → Start server → Open tunnel → Use from local machine

---

## Step 0 — Verify GPU

In [1]:
# Fix the torch/transformers version mismatch
!pip install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q
!pip install -U transformers accelerate fastapi uvicorn pyngrok -q
print("✅ All upgraded — now restart runtime!")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 83.3 MB/s eta 0:00:00
✅ All upgraded — now restart runtime!


In [1]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), '❌ No GPU! Runtime → Change runtime type → T4 GPU'
props = torch.cuda.get_device_properties(0)
gpu_mem = getattr(props, 'total_memory', getattr(props, 'total_mem', 0)) / 1e9
print(f'\n✅ GPU: {torch.cuda.get_device_name(0)} ({gpu_mem:.1f} GB)')

Mon May 11 16:09:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 1 — Install dependencies

In [2]:
!pip install fastapi uvicorn pyngrok accelerate -q
import transformers, fastapi
print(f'✅ transformers={transformers.__version__} fastapi={fastapi.__version__}')

✅ transformers=5.8.0 fastapi=0.136.1


## Step 2 — Configuration

In [11]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
NGROK_TOKEN = '30WWgMwDaKOdaiv07bhUdv3LB01_71aisn3b7o7tLudkqowe9'  # ← paste your token from https://dashboard.ngrok.com/get-started/your-authtoken
PORT = 8000
MAX_MODEL_LEN = 32768  # Qwen2.5-7B supports up to 128K

API_KEY = 'career-ops-colab'

if not NGROK_TOKEN:
    try:
        from google.colab import userdata
        NGROK_TOKEN = userdata.get('NGROK_TOKEN')
        print('✅ Loaded NGROK_TOKEN from Colab Secrets')
    except: pass
if not NGROK_TOKEN:
    raise ValueError('❌ Set NGROK_TOKEN above or in Colab Secrets')
print(f'📋 Model: {MODEL_NAME} | Port: {PORT}')

📋 Model: Qwen/Qwen2.5-3B-Instruct | Port: 8000


## Step 3 — Start the OpenAI-compatible server

In [14]:
import subprocess, time, requests, os

server_code = f'''
import json, os, time, uuid
from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import JSONResponse
import uvicorn
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL = "{MODEL_NAME}"
KEY = "{API_KEY}"

print(f"Loading {{MODEL}}...")
tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
mdl = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True)
print("Model loaded!")

app = FastAPI()

@app.get("/health")
async def health(): return {{"status": "ok"}}

@app.get("/v1/models")
async def models(): return {{"data": [{{"id": MODEL, "object": "model"}}]}}

@app.post("/v1/chat/completions")
async def chat(request: Request):
    auth = request.headers.get("Authorization", "")
    if KEY and auth != f"Bearer {{KEY}}": raise HTTPException(401, "Invalid API key")
    body = await request.json()
    msgs = body.get("messages", [])
    max_tok = body.get("max_tokens", {MAX_MODEL_LEN})
    temp = max(body.get("temperature", 0.7), 0.01)
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tok(text, return_tensors="pt").to(mdl.device)
    ilen = inp["input_ids"].shape[1]
    with torch.no_grad():
        out = mdl.generate(**inp, max_new_tokens=max_tok, temperature=temp, do_sample=True, pad_token_id=tok.eos_token_id)
    new = out[0][ilen:]
    reply = tok.decode(new, skip_special_tokens=True)
    return JSONResponse({{"id": f"chatcmpl-{{uuid.uuid4().hex[:8]}}", "object": "chat.completion", "created": int(time.time()), "model": MODEL, "choices": [{{"index": 0, "message": {{"role": "assistant", "content": reply}}, "finish_reason": "stop"}}], "usage": {{"prompt_tokens": ilen, "completion_tokens": len(new), "total_tokens": ilen+len(new)}}}})

uvicorn.run(app, host="0.0.0.0", port={PORT})
'''

with open('/tmp/llm_server.py', 'w') as f: f.write(server_code)
server_process = subprocess.Popen(['python', '/tmp/llm_server.py'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

print(f'🚀 Starting server with {MODEL_NAME}...')
print('   (first run downloads model — may take 5-10 min)\n')
start = time.time()
ready = False
while time.time() - start < 600:
    try:
        if requests.get(f'http://localhost:{PORT}/health', timeout=2).status_code == 200:
            ready = True; break
    except: pass
    if server_process.poll() is not None:
        raise RuntimeError(f'❌ Server crashed:\n{server_process.stdout.read().decode()[-2000:]}')
    e = int(time.time()-start)
    if e % 30 == 0 and e > 0: print(f'   ... loading ({e}s)')
    time.sleep(5)
if not ready: raise RuntimeError('❌ Server timeout')
print(f'\n✅ Server ready! ({int(time.time()-start)}s)')

🚀 Starting server with Qwen/Qwen2.5-3B-Instruct...
   (first run downloads model — may take 5-10 min)

   ... loading (30s)

✅ Server ready! (50s)


In [13]:
# Kill anything running on port 8000
!fuser -k 8000/tcp


8000/tcp:             6254


## Step 4 — Open ngrok tunnel

In [15]:
from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_TOKEN
tunnel = ngrok.connect(PORT, 'http')
public_url = tunnel.public_url
print('═'*60)
print('  🌐 YOUR LLM SERVER IS LIVE!')
print('═'*60)
print(f'  URL:   {public_url}')
print(f'  Key:   {API_KEY}')
print(f'  Model: {MODEL_NAME}')
print('─'*60)
print('  📋 Run on your local machine:')
print(f'  node local-eval.mjs --base {public_url} --model {MODEL_NAME} --file ./jds/job.txt')
print('─'*60)
print('  Or add to .env:')
print(f'  LOCAL_LLM_BASE_URL={public_url}')
print(f'  LOCAL_LLM_API_KEY={API_KEY}')
print(f'  LOCAL_LLM_MODEL={MODEL_NAME}')
print('═'*60)

════════════════════════════════════════════════════════════
  🌐 YOUR LLM SERVER IS LIVE!
════════════════════════════════════════════════════════════
  URL:   https://b010-34-7-80-91.ngrok-free.app
  Key:   career-ops-colab
  Model: Qwen/Qwen2.5-3B-Instruct
────────────────────────────────────────────────────────────
  📋 Run on your local machine:
  node local-eval.mjs --base https://b010-34-7-80-91.ngrok-free.app --model Qwen/Qwen2.5-3B-Instruct --file ./jds/job.txt
────────────────────────────────────────────────────────────
  Or add to .env:
  LOCAL_LLM_BASE_URL=https://b010-34-7-80-91.ngrok-free.app
  LOCAL_LLM_API_KEY=career-ops-colab
  LOCAL_LLM_MODEL=Qwen/Qwen2.5-3B-Instruct
════════════════════════════════════════════════════════════


In [10]:
# Run this in a new Colab cell to see the error
try:
    # Try to read the last few lines of output
    print(server_process.stdout.read1(8192).decode())
except Exception as e:
    print(f"Could not read logs: {e}")


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading Qwen/Qwen2.5-7B-Instruct...
Loading weights:  91%|█████


## Step 5 — Test

In [6]:
import requests
r = requests.post(f'{public_url}/v1/chat/completions', headers={'Authorization': f'Bearer {API_KEY}', 'Content-Type': 'application/json'}, json={'model': MODEL_NAME, 'messages': [{'role': 'user', 'content': 'Say hello in one word'}], 'max_tokens': 20, 'temperature': 0.3}, timeout=60)
if r.status_code == 200:
    print(f'✅ Working! Reply: {r.json()["choices"][0]["message"]["content"]}')
else:
    print(f'❌ Error {r.status_code}: {r.text[:300]}')

✅ Working! Reply: Hi!


## Step 6 — Keep alive
> Keep this running. Colab may disconnect after ~90 min of inactivity.

In [ ]:
import time
from datetime import datetime
print(f'🔄 Keep-alive started. URL: {public_url}\n')
while True:
    try: s = '✅' if requests.get(f'http://localhost:{PORT}/health', timeout=5).status_code == 200 else '⚠️'
    except: s = '❌'
    print(f'[{datetime.now().strftime("%H:%M")}] {s} | {public_url}')
    if server_process.poll() is not None: print('❌ Server stopped!'); break
    time.sleep(300)